In [87]:
Servr Connection

SyntaxError: invalid syntax (1166882337.py, line 1)

In [113]:
import pyodbc

pyodbc.drivers()


['SQL Server',
 'SQL Server Native Client RDA 11.0',
 'ODBC Driver 17 for SQL Server',
 'ODBC Driver 18 for SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)']

In [114]:
import pandas as pd
from sqlalchemy import create_engine, URL, text

server = r"AHMED-PCC\SQLEXPRESS"
database = "AdventureWorks2022"
driver = "ODBC Driver 18 for SQL Server"

connection_url = URL.create(
    "mssql+pyodbc",
    host=server,
    database=database,
    query={
        "driver": driver,
        "trusted_connection": "yes",
        "TrustServerCertificate": "yes"
    }
)

engine = create_engine(connection_url)

In [115]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DB_NAME() AS DatabaseName")
    )
    print(result.fetchone())

('AdventureWorks2022',)


In [116]:
first of data

SyntaxError: invalid syntax (269445289.py, line 1)

In [117]:
query = """
SELECT
    s.name AS SchemaName,
    t.name AS TableName
FROM sys.tables AS t
JOIN sys.schemas AS s
    ON t.schema_id = s.schema_id
ORDER BY
    s.name,
    t.name;
"""

with engine.connect() as connection:
    tables = pd.read_sql_query(query, connection)

display(tables)

,SchemaName,TableName
0,dbo,AWBuildVersion
1,dbo,DatabaseLog
2,dbo,ErrorLog
3,HumanResources,Department
4,HumanResources,Employee
...,...,...
66,Sales,SalesTerritoryHistory
67,Sales,ShoppingCartItem
68,Sales,SpecialOffer
69,Sales,SpecialOfferProduct


In [118]:
query = """
SELECT
    s.name AS SchemaName,
    t.name AS TableName,
    SUM(p.row_count) AS TotalRows
FROM sys.tables AS t
JOIN sys.schemas AS s
    ON t.schema_id = s.schema_id
LEFT JOIN sys.dm_db_partition_stats AS p
    ON t.object_id = p.object_id
    AND p.index_id IN (0, 1)
GROUP BY
    s.name,
    t.name
ORDER BY
    TotalRows DESC;
"""

with engine.connect() as connection:
    row_counts = pd.read_sql_query(query, connection)

display(row_counts)

,SchemaName,TableName,TotalRows
0,Sales,SalesOrderDetail,121317
1,Production,TransactionHistory,113443
2,Production,TransactionHistoryArchive,89253
3,Production,WorkOrder,72591
4,Production,WorkOrderRouting,67131
...,...,...,...
66,Person,PhoneNumberType,3
67,Sales,ShoppingCartItem,3
68,HumanResources,Shift,3
69,dbo,AWBuildVersion,1


In [119]:
query = """
SELECT
    s.name AS SchemaName,
    t.name AS TableName,
    c.name AS ColumnName,
    ty.name AS DataType,
    c.max_length AS MaxLength,
    c.is_nullable AS IsNullable
FROM sys.tables AS t
JOIN sys.schemas AS s
    ON t.schema_id = s.schema_id
JOIN sys.columns AS c
    ON t.object_id = c.object_id
JOIN sys.types AS ty
    ON c.user_type_id = ty.user_type_id
ORDER BY
    s.name,
    t.name,
    c.column_id;
"""

with engine.connect() as connection:
    columns = pd.read_sql_query(query, connection)

display(columns)

,SchemaName,TableName,ColumnName,DataType,MaxLength,IsNullable
0,dbo,AWBuildVersion,SystemInformationID,tinyint,1,False
1,dbo,AWBuildVersion,Database Version,nvarchar,50,False
2,dbo,AWBuildVersion,VersionDate,datetime,8,False
3,dbo,AWBuildVersion,ModifiedDate,datetime,8,False
4,dbo,DatabaseLog,DatabaseLogID,int,4,False
...,...,...,...,...,...,...
481,Sales,Store,Name,Name,100,False
482,Sales,Store,SalesPersonID,int,4,True
483,Sales,Store,Demographics,xml,-1,True
484,Sales,Store,rowguid,uniqueidentifier,16,False


In [120]:
query = """
SELECT
    fk.name AS ForeignKeyName,
    SCHEMA_NAME(pt.schema_id) AS ParentSchema,
    pt.name AS ParentTable,
    pc.name AS ParentColumn,
    SCHEMA_NAME(rt.schema_id) AS ReferencedSchema,
    rt.name AS ReferencedTable,
    rc.name AS ReferencedColumn
FROM sys.foreign_keys AS fk
JOIN sys.foreign_key_columns AS fkc
    ON fk.object_id = fkc.constraint_object_id
JOIN sys.tables AS pt
    ON fkc.parent_object_id = pt.object_id
JOIN sys.columns AS pc
    ON fkc.parent_object_id = pc.object_id
    AND fkc.parent_column_id = pc.column_id
JOIN sys.tables AS rt
    ON fkc.referenced_object_id = rt.object_id
JOIN sys.columns AS rc
    ON fkc.referenced_object_id = rc.object_id
    AND fkc.referenced_column_id = rc.column_id
ORDER BY
    ParentSchema,
    ParentTable;
"""

with engine.connect() as connection:
    relationships = pd.read_sql_query(query, connection)

display(relationships)

,ForeignKeyName,ParentSchema,ParentTable,ParentColumn,ReferencedSchema,ReferencedTable,ReferencedColumn
0,FK_Employee_Person_BusinessEntityID,HumanResources,Employee,BusinessEntityID,Person,Person,BusinessEntityID
1,FK_EmployeeDepartmentHistory_Employee_Business...,HumanResources,EmployeeDepartmentHistory,BusinessEntityID,HumanResources,Employee,BusinessEntityID
2,FK_EmployeeDepartmentHistory_Department_Depart...,HumanResources,EmployeeDepartmentHistory,DepartmentID,HumanResources,Department,DepartmentID
3,FK_EmployeeDepartmentHistory_Shift_ShiftID,HumanResources,EmployeeDepartmentHistory,ShiftID,HumanResources,Shift,ShiftID
4,FK_EmployeePayHistory_Employee_BusinessEntityID,HumanResources,EmployeePayHistory,BusinessEntityID,HumanResources,Employee,BusinessEntityID
...,...,...,...,...,...,...,...
86,FK_ShoppingCartItem_Product_ProductID,Sales,ShoppingCartItem,ProductID,Production,Product,ProductID
87,FK_SpecialOfferProduct_SpecialOffer_SpecialOff...,Sales,SpecialOfferProduct,SpecialOfferID,Sales,SpecialOffer,SpecialOfferID
88,FK_SpecialOfferProduct_Product_ProductID,Sales,SpecialOfferProduct,ProductID,Production,Product,ProductID
89,FK_Store_BusinessEntity_BusinessEntityID,Sales,Store,BusinessEntityID,Person,BusinessEntity,BusinessEntityID


In [121]:
query = """
SELECT TOP (10) *
FROM Sales.SalesOrderHeader;
"""

with engine.connect() as connection:
    sample = pd.read_sql_query(query, connection)

display(sample)

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,Comment,rowguid,ModifiedDate
0,43659,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43659,PO522145787,10-4020-000676,...,16281,105041Vi84182,NaN,20565.6206,1971.5149,616.0984,23153.2339,None,79B65321-39CA-4115-9CBA-8FE0903E12E6,2011-06-07
1,43660,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43660,PO18850127500,10-4020-000117,...,5618,115213Vi29411,NaN,1294.2529,124.2483,38.8276,1457.3288,None,738DC42D-D03B-48A1-9822-F95A67EA7389,2011-06-07
2,43661,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43661,PO18473189620,10-4020-000442,...,1346,85274Vi6854,4.0,32726.4786,3153.7696,985.5530,36865.8012,None,D91B9131-18A4-4A11-BC3A-90B6F53E9D74,2011-06-07
3,43662,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43662,PO18444174044,10-4020-000227,...,10456,125295Vi53935,4.0,28832.5289,2775.1646,867.2389,32474.9324,None,4A1ECFC0-CC3A-4740-B028-1C50BB48711C,2011-06-07
4,43663,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43663,PO18009186470,10-4020-000510,...,4322,45303Vi22691,NaN,419.4589,40.2681,12.5838,472.3108,None,9B1E7A40-6AE0-4AD3-811C-A64951857C4B,2011-06-07
5,43664,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43664,PO16617121983,10-4020-000397,...,806,95555Vi4081,NaN,24432.6088,2344.9921,732.8100,27510.4109,None,22A8A5DA-8C22-42AD-9241-839489B6EF0D,2011-06-07
6,43665,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43665,PO16588191572,10-4020-000146,...,15232,35568Vi78804,NaN,14352.7713,1375.9427,429.9821,16158.6961,None,5602C304-853C-43D7-9E79-76E320D476CF,2011-06-07
7,43666,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43666,PO16008173883,10-4020-000511,...,13349,105623Vi69217,NaN,5056.4896,486.3747,151.9921,5694.8564,None,E2A90057-1366-4487-8A7E-8085845FF770,2011-06-07
8,43667,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43667,PO15428132599,10-4020-000646,...,10370,55680Vi53503,NaN,6107.0820,586.1203,183.1626,6876.3649,None,86D5237D-432D-4B21-8ABC-671942F5789D,2011-06-07
9,43668,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43668,PO14732180295,10-4020-000514,...,1566,85817Vi8045,4.0,35944.1562,3461.7654,1081.8017,40487.7233,None,281CC355-D538-494E-9B44-461B36A826C6,2011-06-07


In [122]:
query = """
SELECT *
FROM Sales.SalesOrderHeader;
"""

with engine.connect() as connection:
    sales_orders = pd.read_sql_query(query, connection)

display(sales_orders.head())

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,Comment,rowguid,ModifiedDate
0,43659,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43659,PO522145787,10-4020-000676,...,16281.0,105041Vi84182,NaN,20565.6206,1971.5149,616.0984,23153.2339,None,79B65321-39CA-4115-9CBA-8FE0903E12E6,2011-06-07
1,43660,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43660,PO18850127500,10-4020-000117,...,5618.0,115213Vi29411,NaN,1294.2529,124.2483,38.8276,1457.3288,None,738DC42D-D03B-48A1-9822-F95A67EA7389,2011-06-07
2,43661,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43661,PO18473189620,10-4020-000442,...,1346.0,85274Vi6854,4.0,32726.4786,3153.7696,985.5530,36865.8012,None,D91B9131-18A4-4A11-BC3A-90B6F53E9D74,2011-06-07
3,43662,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43662,PO18444174044,10-4020-000227,...,10456.0,125295Vi53935,4.0,28832.5289,2775.1646,867.2389,32474.9324,None,4A1ECFC0-CC3A-4740-B028-1C50BB48711C,2011-06-07
4,43663,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43663,PO18009186470,10-4020-000510,...,4322.0,45303Vi22691,NaN,419.4589,40.2681,12.5838,472.3108,None,9B1E7A40-6AE0-4AD3-811C-A64951857C4B,2011-06-07


In [123]:
understanding data

SyntaxError: invalid syntax (2892393585.py, line 1)

In [ ]:
%pip install openpyxl

In [ ]:

from sqlalchemy import text
import pandas as pd

# 1. Get all schemas and tables
tables_query = """
SELECT
    s.name AS SchemaName,
    t.name AS TableName,
    SUM(ps.row_count) AS TotalRows
FROM sys.tables t
JOIN sys.schemas s
    ON t.schema_id = s.schema_id
LEFT JOIN sys.dm_db_partition_stats ps
    ON t.object_id = ps.object_id
    AND ps.index_id IN (0, 1)
GROUP BY s.name, t.name
ORDER BY s.name, t.name;
"""

# 2. Get all columns and their data types
columns_query = """
SELECT
    s.name AS SchemaName,
    t.name AS TableName,
    c.name AS ColumnName,
    ty.name AS DataType,
    c.max_length AS MaxLength,
    c.is_nullable AS IsNullable,
    c.column_id AS ColumnOrder
FROM sys.tables t
JOIN sys.schemas s
    ON t.schema_id = s.schema_id
JOIN sys.columns c
    ON t.object_id = c.object_id
JOIN sys.types ty
    ON c.user_type_id = ty.user_type_id
ORDER BY s.name, t.name, c.column_id;
"""

# 3. Get primary keys
pk_query = """
SELECT
    s.name AS SchemaName,
    t.name AS TableName,
    c.name AS ColumnName,
    kc.name AS PrimaryKeyName
FROM sys.key_constraints kc
JOIN sys.tables t
    ON kc.parent_object_id = t.object_id
JOIN sys.schemas s
    ON t.schema_id = s.schema_id
JOIN sys.index_columns ic
    ON kc.parent_object_id = ic.object_id
    AND kc.unique_index_id = ic.index_id
JOIN sys.columns c
    ON ic.object_id = c.object_id
    AND ic.column_id = c.column_id
WHERE kc.type = 'PK'
ORDER BY s.name, t.name, ic.key_ordinal;
"""

# 4. Get foreign keys
fk_query = """
SELECT
    fk.name AS ForeignKeyName,
    SCHEMA_NAME(pt.schema_id) AS ParentSchema,
    pt.name AS ParentTable,
    pc.name AS ParentColumn,
    SCHEMA_NAME(rt.schema_id) AS ReferencedSchema,
    rt.name AS ReferencedTable,
    rc.name AS ReferencedColumn
FROM sys.foreign_keys fk
JOIN sys.foreign_key_columns fkc
    ON fk.object_id = fkc.constraint_object_id
JOIN sys.tables pt
    ON fkc.parent_object_id = pt.object_id
JOIN sys.columns pc
    ON fkc.parent_object_id = pc.object_id
    AND fkc.parent_column_id = pc.column_id
JOIN sys.tables rt
    ON fkc.referenced_object_id = rt.object_id
JOIN sys.columns rc
    ON fkc.referenced_object_id = rc.object_id
    AND fkc.referenced_column_id = rc.column_id
ORDER BY ParentSchema, ParentTable, fk.name;
"""

# Run all queries
with engine.connect() as conn:
    tables_df = pd.read_sql_query(text(tables_query), conn)
    columns_df = pd.read_sql_query(text(columns_query), conn)
    primary_keys_df = pd.read_sql_query(text(pk_query), conn)
    foreign_keys_df = pd.read_sql_query(text(fk_query), conn)

print("=== TABLES AND ROW COUNTS ===")
display(tables_df)

print("=== COLUMNS ===")
display(columns_df)

print("=== PRIMARY KEYS ===")
display(primary_keys_df)

print("=== FOREIGN KEYS ===")
display(foreign_keys_df)


In [ ]:
with pd.ExcelWriter("AdventureWorks_Structure.xlsx") as writer:
    tables_df.to_excel(writer, sheet_name="Tables", index=False)
    columns_df.to_excel(writer, sheet_name="Columns", index=False)
    primary_keys_df.to_excel(writer, sheet_name="PrimaryKeys", index=False)
    foreign_keys_df.to_excel(writer, sheet_name="ForeignKeys", index=False)

In [ ]:
Cleaning
SalesOrderHeader جدول

In [ ]:
query = """
SELECT *
FROM Sales.SalesOrderHeader;
"""

with engine.connect() as connection:
    sales_orders = pd.read_sql_query(query, connection)
df = sales_orders
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
import pandas as pd
print("عدد القيم المفقودة قبل التنظيف:")
print(df.isnull().sum())
# 1. إزالة العمود غير المفيد تماماً (Comment)
df.drop(columns=['Comment'], inplace=True)

# 2. معالجة SalesPersonID و PurchaseOrderNumber للطلبات الأونلاين
# نستخدم fillna لاستبدال القيم المفقودة
df['SalesPersonID'] = df['SalesPersonID'].fillna(0).astype(int)
df['PurchaseOrderNumber'] = df['PurchaseOrderNumber'].fillna('N/A')

# 3. معالجة بيانات بطاقات الائتمان (Credit Card)
df['CreditCardID'] = df['CreditCardID'].fillna(0).astype(int)
df['CreditCardApprovalCode'] = df['CreditCardApprovalCode'].fillna('N/A')

# 4. معالجة أسعار تحويل العملات (CurrencyRateID)
df['CurrencyRateID'] = df['CurrencyRateID'].fillna(0).astype(int)

# 5. التأكد من معالجة كل الـ Nulls في الجدول
print("عدد القيم المفقودة بعد التنظيف:")
print(df.isnull().sum())

In [124]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   BusinessEntityID     290 non-null    int64         
 1   JobTitle             290 non-null    str           
 2   BirthDate            290 non-null    datetime64[s] 
 3   MaritalStatus        290 non-null    str           
 4   Gender               290 non-null    str           
 5   HireDate             290 non-null    datetime64[s] 
 6   SalariedFlag         290 non-null    bool          
 7   VacationHours        290 non-null    int64         
 8   SickLeaveHours       290 non-null    int64         
 9   CurrentFlag          290 non-null    bool          
 10  ModifiedDate         290 non-null    datetime64[us]
 11  Is_Current_Employee  290 non-null    bool          
dtypes: bool(3), datetime64[s](2), datetime64[us](1), int64(3), str(3)
memory usage: 21.4 KB


In [125]:
df.shape

(290, 12)

In [126]:
data types

SyntaxError: invalid syntax (609214733.py, line 1)

In [127]:
import pandas as pd

# 1. طباعة أنواع البيانات الحالية وحجم الذاكرة
print("--- أنواع البيانات الحالية ---")
print(df.dtypes)

# 2. تحويل التواريخ لـ datetime (لو لم تكن كذلك)
date_cols = ['OrderDate', 'DueDate', 'ShipDate', 'ModifiedDate']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# 3. التأكد من تحويل المبالغ المالية لـ float
money_cols = ['SubTotal', 'TaxAmt', 'Freight', 'TotalDue']
for col in money_cols:
    df[col] = df[col].astype(float)

# 4. التأكد من سلامة المعاملات الحسابية (Business Validation Check)
# التأكد من أن TotalDue يساوي فعلياً (SubTotal + TaxAmt + Freight)
df['Calculated_Total'] = df['SubTotal'] + df['TaxAmt'] + df['Freight']
# مقارنة مع TotalDue لمعرفة هل يوجد أي فرق
diff = (df['TotalDue'] - df['Calculated_Total']).abs().max()
print(f"\nأقصى فرق بين TotalDue والمحسوب: {diff}")

# حذف العمود الموقت بعد التأكد
df.drop(columns=['Calculated_Total'], inplace=True)

--- أنواع البيانات الحالية ---
BusinessEntityID                int64
JobTitle                          str
BirthDate               datetime64[s]
MaritalStatus                     str
Gender                            str
HireDate                datetime64[s]
SalariedFlag                     bool
VacationHours                   int64
SickLeaveHours                  int64
CurrentFlag                      bool
ModifiedDate           datetime64[us]
Is_Current_Employee              bool
dtype: object


KeyError: 'OrderDate'

In [ ]:
# تقريب القيم المالية لرقمين عشريين في التقارير
df['TotalDue'] = df['TotalDue'].round(2)

In [ ]:
SalesOrderDetail 
Cleaning 

In [ ]:
query_detail = "SELECT * FROM Sales.SalesOrderDetail;"
with engine.connect() as connection:
    df_detail = pd.read_sql_query(query_detail, connection)

In [ ]:
df_detail.info()

In [128]:
df_detail.head()

,SalesOrderID,SalesOrderDetailID,CarrierTrackingNumber,OrderQty,ProductID,SpecialOfferID,UnitPrice,UnitPriceDiscount,LineTotal,ModifiedDate
0,43659,1,4911-403C-98,1,776,1,2024.994,0.0,2024.994,2011-05-31
1,43659,2,4911-403C-98,3,777,1,2024.994,0.0,6074.982,2011-05-31
2,43659,3,4911-403C-98,1,778,1,2024.994,0.0,2024.994,2011-05-31
3,43659,4,4911-403C-98,1,771,1,2039.994,0.0,2039.994,2011-05-31
4,43659,5,4911-403C-98,1,772,1,2039.994,0.0,2039.994,2011-05-31


In [129]:
import pandas as pd
import numpy as np
# ==========================================
# 1. إزالة الأعمدة التقنية غير الضرورية
# ==========================================
if 'rowguid' in df_detail.columns:
    df_detail.drop(columns=['rowguid'], inplace=True)

# ==========================================
# 2. معالجة القيم المفقودة (Nulls)
# ==========================================
# استبدال رقم التتبع المفقود بـ 'N/A'
df_detail['CarrierTrackingNumber'] = df_detail['CarrierTrackingNumber'].fillna('N/A')

# ==========================================
# 3. ضبط أنواع البيانات (Data Types)
# ==========================================
df_detail['ModifiedDate'] = pd.to_datetime(df_detail['ModifiedDate'])

money_cols = ['UnitPrice', 'UnitPriceDiscount', 'LineTotal']
for col in money_cols:
    df_detail[col] = df_detail[col].astype(float)

df_detail['OrderQty'] = df_detail['OrderQty'].astype(int)

In [130]:
df_detail.isnull().sum()

SalesOrderID             0
SalesOrderDetailID       0
CarrierTrackingNumber    0
OrderQty                 0
ProductID                0
SpecialOfferID           0
UnitPrice                0
UnitPriceDiscount        0
LineTotal                0
ModifiedDate             0
dtype: int64

In [131]:
df_detail.duplicated().sum()

np.int64(0)

In [132]:
# حساب إجمالي السطر المتوقع بناءً على الكمية، السعر، والخصم
calculated_line_total = (
    df_detail['OrderQty'] * df_detail['UnitPrice'] * (1 - df_detail['UnitPriceDiscount'])
)

# حساب أقصى فرق بين القيمة الحسابية والمسجلة
max_diff = (df_detail['LineTotal'] - calculated_line_total).abs().max()

duplicate_ids = df_detail['SalesOrderDetailID'].duplicated().sum()
invalid_qty = (df_detail['OrderQty'] <= 0).sum()
invalid_price = (df_detail['UnitPrice'] < 0).sum()

print("=== تقرير تنظيف واستكشاف SalesOrderDetail ===")
print(f"1. عدد القيم المفقودة (Nulls):\n{df_detail.isnull().sum().sum()}")
print(f"2. أقصى فرق حسابي في LineTotal: {max_diff}")
print(f"3. عدد الـ SalesOrderDetailID المكررة: {duplicate_ids}")
print(f"4. عدد الصفوف ذات الكميات غير الصالحة (<= 0): {invalid_qty}")
print(f"5. عدد الصفوف ذات الأسعار السالبة (< 0): {invalid_price}")

# عرض عينة بسيطة من البيانات بعد التنظيف
df_detail.head()

=== تقرير تنظيف واستكشاف SalesOrderDetail ===
1. عدد القيم المفقودة (Nulls):
0
2. أقصى فرق حسابي في LineTotal: 3.637978807091713e-12
3. عدد الـ SalesOrderDetailID المكررة: 0
4. عدد الصفوف ذات الكميات غير الصالحة (<= 0): 0
5. عدد الصفوف ذات الأسعار السالبة (< 0): 0


,SalesOrderID,SalesOrderDetailID,CarrierTrackingNumber,OrderQty,ProductID,SpecialOfferID,UnitPrice,UnitPriceDiscount,LineTotal,ModifiedDate
0,43659,1,4911-403C-98,1,776,1,2024.994,0.0,2024.994,2011-05-31
1,43659,2,4911-403C-98,3,777,1,2024.994,0.0,6074.982,2011-05-31
2,43659,3,4911-403C-98,1,778,1,2024.994,0.0,2024.994,2011-05-31
3,43659,4,4911-403C-98,1,771,1,2039.994,0.0,2039.994,2011-05-31
4,43659,5,4911-403C-98,1,772,1,2039.994,0.0,2039.994,2011-05-31


In [133]:
تلت جداول كمان 

SyntaxError: invalid syntax (1460334383.py, line 1)

In [134]:
query_product = "SELECT * FROM Production.Product;"
query_subcat = "SELECT * FROM Production.ProductSubcategory;"
query_cat = "SELECT * FROM Production.ProductCategory;"

with engine.connect() as connection:
    df_product = pd.read_sql_query(query_product, connection)
    df_subcat = pd.read_sql_query(query_subcat, connection)
    df_cat = pd.read_sql_query(query_cat, connection)

In [167]:
df_product.isnull().sum()

ProductID                 0
Name                      0
ProductNumber             0
MakeFlag                  0
FinishedGoodsFlag         0
Color                     0
SafetyStockLevel          0
ReorderPoint              0
StandardCost              0
ListPrice                 0
Size                      0
Weight                    0
DaysToManufacture         0
ProductLine               0
Class                     0
Style                     0
ProductSubcategoryID      0
ProductModelID            0
SellStartDate             0
SellEndDate             406
DiscontinuedDate        504
ModifiedDate              0
Name_Subcategory          0
Name_Category             0
Profit_Margin             0
Product_Type              0
Product_Status            0
dtype: int64

In [136]:
df_product.info()

<class 'pandas.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   ProductID              504 non-null    int64         
 1   Name                   504 non-null    str           
 2   ProductNumber          504 non-null    str           
 3   MakeFlag               504 non-null    bool          
 4   FinishedGoodsFlag      504 non-null    bool          
 5   Color                  256 non-null    str           
 6   SafetyStockLevel       504 non-null    int64         
 7   ReorderPoint           504 non-null    int64         
 8   StandardCost           504 non-null    float64       
 9   ListPrice              504 non-null    float64       
 10  Size                   211 non-null    str           
 11  SizeUnitMeasureCode    176 non-null    str           
 12  WeightUnitMeasureCode  205 non-null    str           
 13  Weight          

In [137]:
import pandas as pd
import numpy as np

# ==========================================
# 1. إزالة الأعمدة التقنية غير الضرورية
# ==========================================
drop_cols = ['rowguid', 'SizeUnitMeasureCode', 'WeightUnitMeasureCode']
df_product.drop(columns=[col for col in drop_cols if col in df_product.columns], inplace=True)

# ==========================================
# 2. معالجة القيم المفقودة (Nulls)
# ==========================================
# معالجة الأوصاف النصية المفقودة
df_product['Color'] = df_product['Color'].fillna('N/A')
df_product['Size'] = df_product['Size'].fillna('N/A')
df_product['Class'] = df_product['Class'].fillna('N/A')
df_product['Style'] = df_product['Style'].fillna('N/A')
df_product['ProductLine'] = df_product['ProductLine'].fillna('N/A')

# معالجة الأوزان والمفاتيح المفقودة
df_product['Weight'] = df_product['Weight'].fillna(0.0)
df_product['ProductSubcategoryID'] = df_product['ProductSubcategoryID'].fillna(0).astype(int)
df_product['ProductModelID'] = df_product['ProductModelID'].fillna(0).astype(int)

# ==========================================
# 3. ضبط أنواع البيانات (Data Types)
# ==========================================
date_cols = ['SellStartDate', 'SellEndDate', 'DiscontinuedDate', 'ModifiedDate']
for col in date_cols:
    if col in df_product.columns:
        df_product[col] = pd.to_datetime(df_product[col])

df_product['StandardCost'] = df_product['StandardCost'].astype(float)
df_product['ListPrice'] = df_product['ListPrice'].astype(float)

In [138]:
df_product.isnull().sum()

ProductID                 0
Name                      0
ProductNumber             0
MakeFlag                  0
FinishedGoodsFlag         0
Color                     0
SafetyStockLevel          0
ReorderPoint              0
StandardCost              0
ListPrice                 0
Size                      0
Weight                    0
DaysToManufacture         0
ProductLine               0
Class                     0
Style                     0
ProductSubcategoryID      0
ProductModelID            0
SellStartDate             0
SellEndDate             406
DiscontinuedDate        504
ModifiedDate              0
dtype: int64

In [139]:
import pandas as pd
import numpy as np
# ==========================================
# 4. دمج التدرج الهرمي للتصنيفات (Merging Categories)
# ==========================================
# أ) دمج الفئات الفرعية مع الفئات الرئيسية
df_categories = pd.merge(
    df_subcat[['ProductSubcategoryID', 'ProductCategoryID', 'Name']],
    df_cat[['ProductCategoryID', 'Name']],
    on='ProductCategoryID',
    how='left',
    suffixes=('_Subcategory', '_Category')
)

# ب) دمج الفئات مع جدول المنتجات الرئيسي
df_product = pd.merge(
    df_product,
    df_categories[['ProductSubcategoryID', 'Name_Subcategory', 'Name_Category']],
    on='ProductSubcategoryID',
    how='left'
)

# تعبئة المنتجات التي ليس لها تصنيف بـ 'Uncategorized'
df_product['Name_Subcategory'] = df_product['Name_Subcategory'].fillna('Uncategorized')
df_product['Name_Category'] = df_product['Name_Category'].fillna('Uncategorized')

# ==========================================
# 5. حساب مؤشرات البيزنس (Feature Engineering & Business Check)
# ==========================================
# حساب الربح المتوقع للقطعة الواحدة
df_product['Profit_Margin'] = df_product['ListPrice'] - df_product['StandardCost']

# تمييز المنتجات القابلة للبيع عن القطع الداخلية
df_product['Product_Type'] = np.where(df_product['FinishedGoodsFlag'] == 1, 'Salable Item', 'Internal Component')

# ==========================================
# 6. التقرير النهائي للجدولين
# ==========================================
print("=== تقرير تنظيف وربط جدول المنتجات Production.Product ===")
print(f"1. عدد الصفوف الكلي: {len(df_product)}")
print(f"2. عدد الـ Nulls المتبقية في جدول المنتجات: {df_product.isnull().sum().sum()}")
print(f"3. عدد المنتجات القابلة للبيع (Salable): {(df_product['FinishedGoodsFlag'] == 1).sum()}")
print(f"4. عدد الأجزاء التصنيعية الداخلية: {(df_product['FinishedGoodsFlag'] == 0).sum()}")

# عرض عينة بسيطة من جدول المنتجات المدمج بعد التنظيف
df_product[['ProductID', 'Name', 'Name_Category', 'Name_Subcategory', 'StandardCost', 'ListPrice', 'Profit_Margin', 'Product_Type']].head()

=== تقرير تنظيف وربط جدول المنتجات Production.Product ===
1. عدد الصفوف الكلي: 504
2. عدد الـ Nulls المتبقية في جدول المنتجات: 910
3. عدد المنتجات القابلة للبيع (Salable): 295
4. عدد الأجزاء التصنيعية الداخلية: 209


,ProductID,Name,Name_Category,Name_Subcategory,StandardCost,ListPrice,Profit_Margin,Product_Type
0,1,Adjustable Race,Uncategorized,Uncategorized,0.0,0.0,0.0,Internal Component
1,2,Bearing Ball,Uncategorized,Uncategorized,0.0,0.0,0.0,Internal Component
2,3,BB Ball Bearing,Uncategorized,Uncategorized,0.0,0.0,0.0,Internal Component
3,4,Headset Ball Bearings,Uncategorized,Uncategorized,0.0,0.0,0.0,Internal Component
4,316,Blade,Uncategorized,Uncategorized,0.0,0.0,0.0,Internal Component


In [140]:
df_product.isnull().sum()

ProductID                 0
Name                      0
ProductNumber             0
MakeFlag                  0
FinishedGoodsFlag         0
Color                     0
SafetyStockLevel          0
ReorderPoint              0
StandardCost              0
ListPrice                 0
Size                      0
Weight                    0
DaysToManufacture         0
ProductLine               0
Class                     0
Style                     0
ProductSubcategoryID      0
ProductModelID            0
SellStartDate             0
SellEndDate             406
DiscontinuedDate        504
ModifiedDate              0
Name_Subcategory          0
Name_Category             0
Profit_Margin             0
Product_Type              0
dtype: int64

In [141]:
import pandas as pd
import numpy as np

# 1. (اختياري) لو حابب تظهر الكلمة باللغة الإنجليزية أو العربية بدلاً من True/False:
df_product['Product_Status'] = np.where(df_product['SellEndDate'].isnull(), 'Active', 'Inactive')

In [142]:
df_product.isnull().sum()

ProductID                 0
Name                      0
ProductNumber             0
MakeFlag                  0
FinishedGoodsFlag         0
Color                     0
SafetyStockLevel          0
ReorderPoint              0
StandardCost              0
ListPrice                 0
Size                      0
Weight                    0
DaysToManufacture         0
ProductLine               0
Class                     0
Style                     0
ProductSubcategoryID      0
ProductModelID            0
SellStartDate             0
SellEndDate             406
DiscontinuedDate        504
ModifiedDate              0
Name_Subcategory          0
Name_Category             0
Profit_Margin             0
Product_Type              0
Product_Status            0
dtype: int64

In [143]:
جدولين كمان 

SyntaxError: invalid syntax (4137054357.py, line 1)

In [144]:
# سحب جدول العملاء وجدول الأشخاص من قاعدة البيانات
query_customer = "SELECT * FROM Sales.Customer;"
query_person = "SELECT * FROM Person.Person;"

with engine.connect() as connection:
    df_customer = pd.read_sql_query(query_customer, connection)
    df_person = pd.read_sql_query(query_person, connection)

print("Customer Shape:", df_customer.shape)
print("Person Shape:", df_person.shape)

Customer Shape: (19820, 7)
Person Shape: (19972, 13)


In [145]:
df_customer.isnull().sum()

CustomerID           0
PersonID           701
StoreID          18484
TerritoryID          0
AccountNumber        0
rowguid              0
ModifiedDate         0
dtype: int64

In [166]:
df_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 19820 entries, 0 to 19819
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   CustomerID     19820 non-null  int64         
 1   PersonID       19820 non-null  int64         
 2   StoreID        19820 non-null  int64         
 3   TerritoryID    19820 non-null  int64         
 4   AccountNumber  19820 non-null  str           
 5   ModifiedDate   19820 non-null  datetime64[us]
 6   Customer_Type  19820 non-null  str           
dtypes: datetime64[us](1), int64(4), str(2)
memory usage: 1.1 MB


In [147]:
df_customer["StoreID"]=df_customer["StoreID"].fillna(0).astype(int)
df_customer["PersonID"]=df_customer["PersonID"].fillna(0).astype(int)

In [148]:
import pandas as pd
import numpy as np

# ==========================================
# تنظيف جدول Sales.Customer
# ==========================================

# 1. حذف الأعمدة التقنية غير الضرورية
if 'rowguid' in df_customer.columns:
    df_customer.drop(columns=['rowguid'], inplace=True)

# 2. معالجة القيم المفقودة وتحويل أنواع البيانات للمفاتيح
df_customer['PersonID'] = df_customer['PersonID'].fillna(0).astype(int)
df_customer['StoreID'] = df_customer['StoreID'].fillna(0).astype(int)
df_customer['TerritoryID'] = df_customer['TerritoryID'].fillna(0).astype(int)
df_customer['ModifiedDate'] = pd.to_datetime(df_customer['ModifiedDate'])

# 3. تحديد نوع العميل (Customer_Type) بناءً على البيزنس logic
# لو PersonID مش بـ 0 يبقى عميل فردي، وإلا يبقى متجر/شركة
df_customer['Customer_Type'] = np.where(
    df_customer['PersonID'] != 0, 
    'Individual Customer', 
    'Store / Reseller'
)

In [149]:
df_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 19820 entries, 0 to 19819
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   CustomerID     19820 non-null  int64         
 1   PersonID       19820 non-null  int64         
 2   StoreID        19820 non-null  int64         
 3   TerritoryID    19820 non-null  int64         
 4   AccountNumber  19820 non-null  str           
 5   ModifiedDate   19820 non-null  datetime64[us]
 6   Customer_Type  19820 non-null  str           
dtypes: datetime64[us](1), int64(4), str(2)
memory usage: 1.1 MB


In [150]:
df_person.isnull().sum()

BusinessEntityID             0
PersonType                   0
NameStyle                    0
Title                    18963
FirstName                    0
MiddleName                8499
LastName                     0
Suffix                   19919
EmailPromotion               0
AdditionalContactInfo    19962
Demographics                 0
rowguid                      0
ModifiedDate                 0
dtype: int64

In [151]:
df_person.head()

,BusinessEntityID,PersonType,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,EmailPromotion,AdditionalContactInfo,Demographics,rowguid,ModifiedDate
0,1,EM,False,NaN,Ken,J,Sánchez,NaN,0,NaN,"<IndividualSurvey xmlns=""http://schemas.micros...",92C4279F-1207-48A3-8448-4636514EB7E2,2009-01-07
1,2,EM,False,NaN,Terri,Lee,Duffy,NaN,1,NaN,"<IndividualSurvey xmlns=""http://schemas.micros...",D8763459-8AA8-47CC-AFF7-C9079AF79033,2008-01-24
2,3,EM,False,NaN,Roberto,NaN,Tamburello,NaN,0,NaN,"<IndividualSurvey xmlns=""http://schemas.micros...",E1A2555E-0828-434B-A33B-6F38136A37DE,2007-11-04
3,4,EM,False,NaN,Rob,NaN,Walters,NaN,0,NaN,"<IndividualSurvey xmlns=""http://schemas.micros...",F2D7CE06-38B3-4357-805B-F4B6B71C01FF,2007-11-28
4,5,EM,False,Ms.,Gail,A,Erickson,NaN,0,NaN,"<IndividualSurvey xmlns=""http://schemas.micros...",F3A3F6B4-AE3B-430C-A754-9F2231BA6FEF,2007-12-30


In [152]:
import pandas as pd

# ==========================================
# تنظيف جدول Person.Person
# ==========================================

# 1. حذف الأعمدة التقنية والثقيلة غير الضرورية
person_drop_cols = ['rowguid', 'AdditionalContactInfo', 'Demographics', 'EmailPromotion', 'NameStyle']
df_person.drop(columns=[col for col in person_drop_cols if col in df_person.columns], inplace=True)

# 2. معالجة القيم المفقودة (Nulls)
df_person['Title'] = df_person['Title'].fillna('N/A')
df_person['Suffix'] = df_person['Suffix'].fillna('N/A')
df_person['MiddleName'] = df_person['MiddleName'].fillna('')  # نص فارغ لسهولة دمج الاسم

# 3. ضبط أنواع البيانات (Data Types)
df_person['ModifiedDate'] = pd.to_datetime(df_person['ModifiedDate'])

# 4. إنشاء عمود الاسم الكامل (Full_Name)
# نستخدم .strip() لإزالة أي مسافات زائدة ناتجة عن MiddleName الفارغ
df_person['Full_Name'] = (
    df_person['FirstName'] + ' ' + 
    df_person['MiddleName'].apply(lambda x: x + ' ' if x != '' else '') + 
    df_person['LastName']
).str.strip()

In [153]:
df_person.head()

,BusinessEntityID,PersonType,Title,FirstName,MiddleName,LastName,Suffix,ModifiedDate,Full_Name
0,1,EM,N/A,Ken,J,Sánchez,N/A,2009-01-07,Ken J Sánchez
1,2,EM,N/A,Terri,Lee,Duffy,N/A,2008-01-24,Terri Lee Duffy
2,3,EM,N/A,Roberto,,Tamburello,N/A,2007-11-04,Roberto Tamburello
3,4,EM,N/A,Rob,,Walters,N/A,2007-11-28,Rob Walters
4,5,EM,Ms.,Gail,A,Erickson,N/A,2007-12-30,Gail A Erickson


In [154]:
df_customer.head()

,CustomerID,PersonID,StoreID,TerritoryID,AccountNumber,ModifiedDate,Customer_Type
0,1,0,934,1,AW00000001,2014-09-12 11:15:07.263,Store / Reseller
1,2,0,1028,1,AW00000002,2014-09-12 11:15:07.263,Store / Reseller
2,3,0,642,4,AW00000003,2014-09-12 11:15:07.263,Store / Reseller
3,4,0,932,4,AW00000004,2014-09-12 11:15:07.263,Store / Reseller
4,5,0,1026,4,AW00000005,2014-09-12 11:15:07.263,Store / Reseller


In [155]:
تنضيف في كود واحد

SyntaxError: invalid syntax (2255723643.py, line 1)

In [156]:
import pandas as pd
import numpy as np

# =========================================================
# 1. تجهيز وتنظيف جدول الأشخاص (df_person)
# =========================================================
# حذف أعمدة غير ضرورية
person_drop_cols = ['rowguid', 'AdditionalContactInfo', 'Demographics', 'EmailPromotion', 'NameStyle']
df_person.drop(columns=[col for col in person_drop_cols if col in df_person.columns], inplace=True)

# معالجة القيم الفاضية والتاريخ
df_person['Title'] = df_person['Title'].fillna('N/A')
df_person['Suffix'] = df_person['Suffix'].fillna('N/A')
df_person['MiddleName'] = df_person['MiddleName'].fillna('')
df_person['ModifiedDate'] = pd.to_datetime(df_person['ModifiedDate'])

# دمج الاسم الأول والأوسط والأخير في عمود واحد لتسهيل القراءة
df_person['Full_Name'] = (
    df_person['FirstName'] + ' ' + 
    df_person['MiddleName'].apply(lambda x: x + ' ' if x != '' else '') + 
    df_person['LastName']
).str.strip()


# =========================================================
# 2. تجهيز وتنظيف جدول العملاء (df_customer)
# =========================================================
# حذف أعمدة غير ضرورية
if 'rowguid' in df_customer.columns:
    df_customer.drop(columns=['rowguid'], inplace=True)

# تحويل المفاتيح والأرقام لأرقام صحيحة بدلاً من فاصلة عائمة
df_customer['PersonID'] = df_customer['PersonID'].fillna(0).astype(int)
df_customer['StoreID'] = df_customer['StoreID'].fillna(0).astype(int)
df_customer['TerritoryID'] = df_customer['TerritoryID'].fillna(0).astype(int)
df_customer['ModifiedDate'] = pd.to_datetime(df_customer['ModifiedDate'])

# إضافة عمود يعرّفك نوع العميل (شخص أم متجر)
df_customer['Customer_Type'] = np.where(
    df_customer['PersonID'] != 0, 
    'Individual Customer', 
    'Store / Reseller'
)


# =========================================================
# 3. عرض عينة من كل جدول منفصل
# =========================================================
print("--- 1. جدول الأشخاص (df_person) ---")
display(df_person[['BusinessEntityID', 'Full_Name', 'PersonType']].head())

print("\n--- 2. جدول العملاء (df_customer) ---")
display(df_customer[['CustomerID', 'AccountNumber', 'Customer_Type', 'PersonID', 'StoreID', 'TerritoryID']].head())

--- 1. جدول الأشخاص (df_person) ---


,BusinessEntityID,Full_Name,PersonType
0,1,Ken J Sánchez,EM
1,2,Terri Lee Duffy,EM
2,3,Roberto Tamburello,EM
3,4,Rob Walters,EM
4,5,Gail A Erickson,EM



--- 2. جدول العملاء (df_customer) ---


,CustomerID,AccountNumber,Customer_Type,PersonID,StoreID,TerritoryID
0,1,AW00000001,Store / Reseller,0,934,1
1,2,AW00000002,Store / Reseller,0,1028,1
2,3,AW00000003,Store / Reseller,0,642,4
3,4,AW00000004,Store / Reseller,0,932,4
4,5,AW00000005,Store / Reseller,0,1026,4


In [157]:
سحب بقيت الحداول

SyntaxError: invalid syntax (544282291.py, line 1)

In [158]:
import pandas as pd

# ==========================================
# استدعاء الجداول المتبقية مع معالجة عمود hierarchyid
# ==========================================

# 1. جدول المناطق الجغرافية (SalesTerritory)
query_territory = "SELECT * FROM Sales.SalesTerritory"
df_territory = pd.read_sql(query_territory, engine)

# 2. جدول مندوبي المبيعات (SalesPerson)
query_sales_person = "SELECT * FROM Sales.SalesPerson"
df_sales_person = pd.read_sql(query_sales_person, engine)

# 3. جدول الموظفين (Employee) - تم اختيار الأعمدة وتجاهل OrganizationNode
query_employee = """
SELECT 
    BusinessEntityID,
    NationalIDNumber,
    LoginID,
    JobTitle,
    BirthDate,
    MaritalStatus,
    Gender,
    HireDate,
    SalariedFlag,
    VacationHours,
    SickLeaveHours,
    CurrentFlag,
    rowguid,
    ModifiedDate
FROM HumanResources.Employee
"""
df_employee = pd.read_sql(query_employee, engine)

# 4. جدول المخزون (ProductInventory)
query_inventory = "SELECT * FROM Production.ProductInventory"
df_inventory = pd.read_sql(query_inventory, engine)

print("✅ تم استدعاء الجداول الأربعة بنجاح!")

✅ تم استدعاء الجداول الأربعة بنجاح!


In [159]:
df_employee.info()

<class 'pandas.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   BusinessEntityID  290 non-null    int64         
 1   NationalIDNumber  290 non-null    str           
 2   LoginID           290 non-null    str           
 3   JobTitle          290 non-null    str           
 4   BirthDate         290 non-null    object        
 5   MaritalStatus     290 non-null    str           
 6   Gender            290 non-null    str           
 7   HireDate          290 non-null    object        
 8   SalariedFlag      290 non-null    bool          
 9   VacationHours     290 non-null    int64         
 10  SickLeaveHours    290 non-null    int64         
 11  CurrentFlag       290 non-null    bool          
 12  rowguid           290 non-null    str           
 13  ModifiedDate      290 non-null    datetime64[us]
dtypes: bool(2), datetime64[us](1), int64(

In [160]:
df_sales_person.info()

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   BusinessEntityID  17 non-null     int64         
 1   TerritoryID       14 non-null     float64       
 2   SalesQuota        14 non-null     float64       
 3   Bonus             17 non-null     float64       
 4   CommissionPct     17 non-null     float64       
 5   SalesYTD          17 non-null     float64       
 6   SalesLastYear     17 non-null     float64       
 7   rowguid           17 non-null     str           
 8   ModifiedDate      17 non-null     datetime64[us]
dtypes: datetime64[us](1), float64(6), int64(1), str(1)
memory usage: 1.3 KB


In [161]:
df_territory

,TerritoryID,Name,CountryRegionCode,Group,SalesYTD,SalesLastYear,CostYTD,CostLastYear,rowguid,ModifiedDate
0,1,Northwest,US,North America,7.887187e+06,3.298694e+06,0.0,0.0,43689A10-E30B-497F-B0DE-11DE20267FF7,2008-04-30
1,2,Northeast,US,North America,2.402177e+06,3.607149e+06,0.0,0.0,00FB7309-96CC-49E2-8363-0A1BA72486F2,2008-04-30
2,3,Central,US,North America,3.072175e+06,3.205014e+06,0.0,0.0,DF6E7FD8-1A8D-468C-B103-ED8ADDB452C1,2008-04-30
3,4,Southwest,US,North America,1.051085e+07,5.366576e+06,0.0,0.0,DC3E9EA0-7950-4431-9428-99DBCBC33865,2008-04-30
4,5,Southeast,US,North America,2.538667e+06,3.925071e+06,0.0,0.0,6DC4165A-5E4C-42D2-809D-4344E0AC75E7,2008-04-30
5,6,Canada,CA,North America,6.771829e+06,5.693989e+06,0.0,0.0,06B4AF8A-1639-476E-9266-110461D66B00,2008-04-30
6,7,France,FR,Europe,4.772398e+06,2.396540e+06,0.0,0.0,BF806804-9B4C-4B07-9D19-706F2E689552,2008-04-30
7,8,Germany,DE,Europe,3.805202e+06,1.307950e+06,0.0,0.0,6D2450DB-8159-414F-A917-E73EE91C38A9,2008-04-30
8,9,Australia,AU,Pacific,5.977815e+06,2.278549e+06,0.0,0.0,602E612E-DFE9-41D9-B894-27E489747885,2008-04-30
9,10,United Kingdom,GB,Europe,5.012905e+06,1.635823e+06,0.0,0.0,05FC7E1F-2DEA-414E-9ECD-09D150516FB5,2008-04-30


In [162]:
Cleaning

NameError: name 'Cleaning' is not defined

In [165]:
df_sales_person.info()

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   BusinessEntityID  17 non-null     int64         
 1   TerritoryID       17 non-null     int64         
 2   SalesQuota        17 non-null     float64       
 3   Bonus             17 non-null     float64       
 4   CommissionPct     17 non-null     float64       
 5   SalesYTD          17 non-null     float64       
 6   SalesLastYear     17 non-null     float64       
 7   ModifiedDate      17 non-null     datetime64[us]
dtypes: datetime64[us](1), float64(5), int64(2)
memory usage: 1.2 KB


In [164]:
# ==========================================
# 1. تنظيف الجداول المتبقية
# ==========================================

# -- تنظيف df_territory
if 'rowguid' in df_territory.columns:
    df_territory.drop(columns=['rowguid'], inplace=True)
df_territory['CostYTD'] = df_territory['CostYTD'].fillna(0.0)
df_territory['CostLastYear'] = df_territory['CostLastYear'].fillna(0.0)
df_territory['ModifiedDate'] = pd.to_datetime(df_territory['ModifiedDate'])

# -- تنظيف df_sales_person
if 'rowguid' in df_sales_person.columns:
    df_sales_person.drop(columns=['rowguid'], inplace=True)
df_sales_person['SalesQuota'] = df_sales_person['SalesQuota'].fillna(0.0)
df_sales_person['Bonus'] = df_sales_person['Bonus'].fillna(0.0)
df_sales_person['CommissionPct'] = df_sales_person['CommissionPct'].fillna(0.0)
df_sales_person['TerritoryID'] = df_sales_person['TerritoryID'].fillna(0).astype(int)
df_sales_person['ModifiedDate'] = pd.to_datetime(df_sales_person['ModifiedDate'])

# -- تنظيف df_employee
emp_drop_cols = ['NationalIDNumber', 'LoginID', 'rowguid']
df_employee.drop(columns=[col for col in emp_drop_cols if col in df_employee.columns], inplace=True)
df_employee['BirthDate'] = pd.to_datetime(df_employee['BirthDate'])
df_employee['HireDate'] = pd.to_datetime(df_employee['HireDate'])
df_employee['ModifiedDate'] = pd.to_datetime(df_employee['ModifiedDate'])
df_employee['Is_Current_Employee'] = df_employee['CurrentFlag'].apply(lambda x: True if x == 1 else False)

# -- تنظيف df_inventory
if 'rowguid' in df_inventory.columns:
    df_inventory.drop(columns=['rowguid'], inplace=True)
df_inventory['Quantity'] = df_inventory['Quantity'].fillna(0).astype(int)
df_inventory['Shelf'] = df_inventory['Shelf'].fillna('N/A')
df_inventory['Bin'] = df_inventory['Bin'].fillna(0).astype(int)
df_inventory['ModifiedDate'] = pd.to_datetime(df_inventory['ModifiedDate'])
df_inventory['Stock_Status'] = df_inventory['Quantity'].apply(
    lambda x: 'Low Stock Alert' if x < 20 else 'Sufficient Stock'
)

# ==========================================
# 2. فحص التكرارات الشامل لكل جداول المشروع الـ 11
# ==========================================

pk_check_map = {
    'df_sales_header': ('SalesOrderID', df_sales_header if 'df_sales_header' in locals() else None),
    'df_sales_detail': ('SalesOrderDetailID', df_sales_detail if 'df_sales_detail' in locals() else None),
    'df_customer': ('CustomerID', df_customer if 'df_customer' in locals() else None),
    'df_person': ('BusinessEntityID', df_person if 'df_person' in locals() else None),
    'df_product': ('ProductID', df_product if 'df_product' in locals() else None),
    'df_territory': ('TerritoryID', df_territory if 'df_territory' in locals() else None),
    'df_sales_person': ('BusinessEntityID', df_sales_person if 'df_sales_person' in locals() else None),
    'df_employee': ('BusinessEntityID', df_employee if 'df_employee' in locals() else None)
}

print("=== 🔍 تقرير فحص التكرارات الشامل ===")
for table_name, (pk_col, df) in pk_check_map.items():
    if df is not None:
        full_row_dups = df.duplicated().sum()
        pk_dups = df[pk_col].duplicated().sum() if pk_col in df.columns else 0
        print(f"📌 {table_name}:")
        print(f"   - تكرار صفوف كاملة: {full_row_dups}")
        print(f"   - تكرار المفتاح الرئيسي ({pk_col}): {pk_dups}")

=== 🔍 تقرير فحص التكرارات الشامل ===
📌 df_customer:
   - تكرار صفوف كاملة: 0
   - تكرار المفتاح الرئيسي (CustomerID): 0
📌 df_person:
   - تكرار صفوف كاملة: 0
   - تكرار المفتاح الرئيسي (BusinessEntityID): 0
📌 df_product:
   - تكرار صفوف كاملة: 0
   - تكرار المفتاح الرئيسي (ProductID): 0
📌 df_territory:
   - تكرار صفوف كاملة: 0
   - تكرار المفتاح الرئيسي (TerritoryID): 0
📌 df_sales_person:
   - تكرار صفوف كاملة: 0
   - تكرار المفتاح الرئيسي (BusinessEntityID): 0
📌 df_employee:
   - تكرار صفوف كاملة: 0
   - تكرار المفتاح الرئيسي (BusinessEntityID): 0


In [168]:
import pandas as pd
from sqlalchemy import create_engine, text

# ==========================================
# 1. إعداد الاتصال بالسيرفر وإنشاء الداتا بيز الجديدة
# ==========================================
SERVER_NAME = r'AHMED-PCC\SQLEXPRESS'  # عدل اسم السيرفر إذا كان مختلفاً لديك (مثلاً '.\SQLEXPRESS')
NEW_DB_NAME = 'AdventureWorks_Cleanned'

# اتصال مبدئي بالسيرفر لإنشاء قاعدة البيانات الجديدة
master_engine = create_engine(
    f"mssql+pyodbc://@{SERVER_NAME}/master?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes",
    isolation_level="AUTOCOMMIT"
)

with master_engine.connect() as conn:
    conn.execute(text(f"IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{NEW_DB_NAME}') CREATE DATABASE {NEW_DB_NAME};"))
    print(f"✅ تم تأكيد/إنشاء قاعدة البيانات: {NEW_DB_NAME}")

# ==========================================
# 2. الاتصال بقاعدة البيانات الجديدة وإنشاء السكيمات الـ 4
# ==========================================
clean_engine = create_engine(
    f"mssql+pyodbc://@{SERVER_NAME}/{NEW_DB_NAME}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes",
    isolation_level="AUTOCOMMIT"
)

schemas = ['Sales', 'Production', 'Person', 'HumanResources']

with clean_engine.connect() as conn:
    for schema in schemas:
        conn.execute(text(f"IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = '{schema}') EXEC('CREATE SCHEMA [{schema}]');"))
    print("✅ تم إنشاء السكيمات الـ 4 بنجاح!")

# ==========================================
# 3. خريطة ربط الـ DataFrames بأسماء المتغيرات الدقيقة
# ==========================================
tables_export_map = {
    # --- Sales Schema ---
    ('Sales', 'SalesOrderHeader'): sales_orders if 'sales_orders' in locals() else None,
    ('Sales', 'SalesOrderDetail'): df_detail if 'df_detail' in locals() else None,
    ('Sales', 'Customer'): df_customer if 'df_customer' in locals() else None,
    ('Sales', 'SalesPerson'): df_sales_person if 'df_sales_person' in locals() else None,
    ('Sales', 'SalesTerritory'): df_territory if 'df_territory' in locals() else None,

    # --- Production Schema ---
    ('Production', 'Product'): df_product if 'df_product' in locals() else None,
    ('Production', 'ProductSubcategory'): df_subcat if 'df_subcat' in locals() else None,
    ('Production', 'ProductCategory'): df_cat if 'df_cat' in locals() else None,
    ('Production', 'ProductInventory'): df_inventory if 'df_inventory' in locals() else None,

    # --- Person Schema ---
    ('Person', 'Person'): df_person if 'df_person' in locals() else None,

    # --- HumanResources Schema ---
    ('HumanResources', 'Employee'): df_employee if 'df_employee' in locals() else None
}

# ==========================================
# 4. تصدير البيانات إلى قاعدة البيانات الجديدة
# ==========================================
print("\n=== 🚀 بدء رفع الجداول النظيفة إلى قاعدة البيانات الجديدة ===")

for (schema_name, table_name), df in tables_export_map.items():
    if df is not None:
        try:
            df.to_sql(
                name=table_name,
                con=clean_engine,
                schema=schema_name,
                if_exists='replace',
                index=False,
                chunksize=1000
            )
            print(f"  🟢 [{schema_name}].[{table_name}] -> تم الرفع بنجاح ({len(df)} صف).")
        except Exception as e:
            print(f"  🔴 [{schema_name}].[{table_name}] -> حدث خطأ أثناء الرفع: {e}")
    else:
        print(f"  ⚠️ [{schema_name}].[{table_name}] -> الـ DataFrame غير موجود في الذاكرة. تأكد من إعادة تشغيل خلية القراءة الخاصة به.")

print("\n🎉 اكتمل رفع جميع الجداول النظيفة إلى قاعدة البيانات الجديدة!")

✅ تم تأكيد/إنشاء قاعدة البيانات: AdventureWorks_Cleanned
✅ تم إنشاء السكيمات الـ 4 بنجاح!

=== 🚀 بدء رفع الجداول النظيفة إلى قاعدة البيانات الجديدة ===
  🟢 [Sales].[SalesOrderHeader] -> تم الرفع بنجاح (31465 صف).
  🟢 [Sales].[SalesOrderDetail] -> تم الرفع بنجاح (121317 صف).
  🟢 [Sales].[Customer] -> تم الرفع بنجاح (19820 صف).
  🟢 [Sales].[SalesPerson] -> تم الرفع بنجاح (17 صف).
  🟢 [Sales].[SalesTerritory] -> تم الرفع بنجاح (10 صف).
  🟢 [Production].[Product] -> تم الرفع بنجاح (504 صف).
  🟢 [Production].[ProductSubcategory] -> تم الرفع بنجاح (37 صف).
  🟢 [Production].[ProductCategory] -> تم الرفع بنجاح (4 صف).
  🟢 [Production].[ProductInventory] -> تم الرفع بنجاح (1069 صف).
  🟢 [Person].[Person] -> تم الرفع بنجاح (19972 صف).
  🟢 [HumanResources].[Employee] -> تم الرفع بنجاح (290 صف).

🎉 اكتمل رفع جميع الجداول النظيفة إلى قاعدة البيانات الجديدة!
